In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install timm editdistance scipy -q

In [ ]:
import zipfile, os

# ── Configure ALL your zip files and their gt txt files here ──────────────
DRIVE   = '/content/drive/MyDrive/GeoAware_project/datasets'
LOCAL   = '/content/data'

# (zip_path, gt_txt_path, output_txt_path, local_extract_folder)
DATASETS = [
    # TRAIN
    (f'{DRIVE}/train/art_train.zip',
     f'{DRIVE}/train/art_train.txt',
     f'{LOCAL}/train/art_train.txt',
     f'{LOCAL}/art_train'),

    (f'{DRIVE}/train/iiit5k_train.zip',
     f'{DRIVE}/train/iiit5k_train.txt',
     f'{LOCAL}/train/iiit5k_train.txt',
     f'{LOCAL}/iiit5k_train'),

    (f'{DRIVE}/train/mjsynth.zip',
     f'{DRIVE}/train/mjsynth_path_label.txt',
     f'{LOCAL}/train/mjsynth_path_label.txt',
     f'{LOCAL}/mjsynth'),

    (f'{DRIVE}/train/total_train.zip',
     f'{DRIVE}/train/total_train.txt',
     f'{LOCAL}/train/totaltext_train_gt.txt',
     f'{LOCAL}/totaltext_train_gt'),

    # TEST
    (f'{DRIVE}/test/art_test.zip',
     f'{DRIVE}/test/art_test_gt.txt',
     f'{LOCAL}/test/art_test_gt.txt',
     f'{LOCAL}/art_test'),

    (f'{DRIVE}/test/iiit5k_test.zip',
     f'{DRIVE}/test/iiit5k_test.txt',
     f'{LOCAL}/test/iiit5k_test.txt',
     f'{LOCAL}/iiit5k_test'),

    (f'{DRIVE}/test/totaltext_test.zip',
     f'{DRIVE}/test/totaltext_test_gt.txt',
     f'{LOCAL}/test/totaltext_test_gt.txt',
     f'{LOCAL}/totaltext_test'),
]
# ──────────────────────────────────────────────────────────────────────────


def find_img_dir(extract_to):
    """Handle zips that extract into a subfolder."""
    entries = os.listdir(extract_to)
    if len(entries) == 1 and os.path.isdir(os.path.join(extract_to, entries[0])):
        return os.path.join(extract_to, entries[0])
    return extract_to


def remap_txt(gt_path, out_path, img_dir):
    remapped = missing = 0
    lines_out = []
    with open(gt_path) as f:
        lines = [l.strip() for l in f if l.strip()]
    for line in lines:
        parts = line.split('\t')
        if len(parts) < 2:
            continue
        filename   = parts[0].replace('\\', '/').split('/')[-1]
        label      = parts[-1].strip()
        local_path = os.path.join(img_dir, filename)
        if os.path.exists(local_path):
            lines_out.append(f"{local_path}\t{label}")
            remapped += 1
        else:
            missing += 1
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, 'w') as f:
        f.write('\n'.join(lines_out))
    return remapped, missing


# ── Main ──────────────────────────────────────────────────────────────────
print("=" * 65)
results = []

for zip_path, gt_path, out_path, extract_to in DATASETS:
    name = os.path.basename(zip_path)

    # Skip missing zips
    if not os.path.exists(zip_path):
        print(f"[SKIP]  {name}  — zip not on Drive")
        results.append((name, None, None))
        continue

    # Unzip (skip if already done)
    os.makedirs(extract_to, exist_ok=True)
    n_existing = len(os.listdir(extract_to))
    if n_existing > 1:
        print(f"[SKIP]  {name}  — already extracted ({n_existing} entries)")
    else:
        print(f"[UNZIP] {name} ...", end=' ', flush=True)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_to)
        print(f"{len(os.listdir(extract_to))} entries")

    # Remap txt
    if not os.path.exists(gt_path):
        print(f"  [SKIP remap] gt txt not found: {gt_path}")
        results.append((name, 0, 0))
        continue

    img_dir = find_img_dir(extract_to)
    remapped, missing = remap_txt(gt_path, out_path, img_dir)
    ok = '✓' if remapped > 0 else '✗'
    print(f"  {ok} remapped={remapped:>6,}  missing={missing:>6,}  → {out_path}")
    results.append((name, remapped, missing))

# Summary
print("\n" + "=" * 65)
print(f"{'Dataset':<35} {'Remapped':>10} {'Missing':>10}")
print("-" * 65)
for name, r, m in results:
    if r is None:
        print(f"{name:<35} {'NOT FOUND':>10}")
    else:
        print(f"{name:<35} {r:>10,} {m:>10,}")
print("=" * 65)

[UNZIP] art_train.zip ... 1 entries
  ✓ remapped=30,520  missing=     0  → /content/data/train/art_train.txt
[UNZIP] iiit5k_train.zip ... 1 entries
  ✓ remapped= 2,000  missing=     0  → /content/data/train/iiit5k_train.txt
[UNZIP] mjsynth.zip ... 1 entries
  ✓ remapped=96,232  missing=38,596  → /content/data/train/mjsynth_path_label.txt
[UNZIP] total_train.zip ... 1 entries
  ✓ remapped= 9,277  missing=     0  → /content/data/train/totaltext_train_gt.txt
[UNZIP] art_test.zip ... 1 entries
  ✓ remapped= 5,388  missing=     0  → /content/data/test/art_test_gt.txt
[UNZIP] iiit5k_test.zip ... 1 entries
  ✓ remapped= 3,000  missing=     0  → /content/data/test/iiit5k_test.txt
[UNZIP] totaltext_test.zip ... 1 entries
  ✓ remapped= 2,211  missing=     0  → /content/data/test/totaltext_test_gt.txt

Dataset                               Remapped    Missing
-----------------------------------------------------------------
art_train.zip                           30,520          0
iiit5k_train.zi

In [ ]:
import os

CHARSET_36 = '0123456789abcdefghijklmnopqrstuvwxyz'

with open('/content/data/train/art_train.txt') as f:
    lines = [l.strip() for l in f if l.strip()]

kept = []
stats = {'empty': 0, 'too_short': 0, 'too_long': 0, 'good': 0}

for line in lines:
    parts = line.split('\t')
    if len(parts) < 2: continue
    path = parts[0].strip()
    raw  = parts[-1].strip().lower()
    gt36 = ''.join(c for c in raw if c in CHARSET_36)

    if not os.path.exists(path):        continue
    if len(gt36) == 0:                  stats['empty']     += 1; continue
    if len(gt36) < 2:                   stats['too_short'] += 1; continue
    if len(gt36) > 15:                  stats['too_long']  += 1; continue

    stats['good'] += 1
    kept.append(f"{path}\t{gt36}")

out_path = '/content/data/train/art_train_clean.txt'
with open(out_path, 'w') as f:
    f.write('\n'.join(kept))

print(f"✓ Saved {stats['good']} samples → {out_path}")
print(f"  Removed: too_short={stats['too_short']} too_long={stats['too_long']} empty={stats['empty']}")

✓ Saved 29368 samples → /content/data/train/art_train_clean.txt
  Removed: too_short=508 too_long=505 empty=139


In [ ]:
import shutil
drive_path = '/content/drive/MyDrive/GeoAware_project/datasets/train/art_train_clean.txt'
os.makedirs(os.path.dirname(drive_path), exist_ok=True)
shutil.copy(out_path, drive_path)
print(f"✓ Backed up to Drive → {drive_path}")

✓ Backed up to Drive → /content/drive/MyDrive/GeoAware_project/datasets/train/art_train_clean.txt


In [ ]:
%%bash
python /content/drive/MyDrive/GeoAware_project/train.py \
  --stage all --no_pretrained --charset 36 \
  --iiit5k_in_stage3 \
  --mjsynth_txt    /content/data/train/mjsynth_path_label.txt \
  --iiit5k_train   /content/data/train/iiit5k_train.txt \
  --art_txt        /content/data/train/art_train.txt \
  --totaltext_txt  /content/data/train/totaltext_train_gt.txt \
  --mjsynth_max 134828 --art_max 36000 --totaltext_max 9000 \
  --batch_size 8 --accum_steps 4 \
  --epochs_s1 12 --epochs_s2 12 --epochs_s3 25 \
  --lr_s1 1e-4 --lr_s2 5e-5 --lr_s3 1e-4 \
  --save_dir /content/drive/MyDrive/GeoAware_project/checkpoints_geoaware_v4 \
  2>&1 | tee /content/drive/MyDrive/GeoAware_project/log_geoaware_v4.txt